# nb

> Learn with code in a notebook (leela or Jupyter): question cards, and code exercises that the kernel checks.

This module brings the tutor into a notebook. `learn` shows a question card in the output of a cell.
`exercise` gives a code exercise, and its tests run on your function in the kernel.
Both use the same `Tutor` and the same `learning` shelf as the web app.

In [ ]:
#| default_exp nb

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

These are the imports of the module.

In [ ]:
#| export
import html, os, traceback, uuid
from pathlib import Path
from fastcore.utils import *
from fastcore.nbio import new_nb, mk_cell, write_nb
from abhyasa.store import Learning
from abhyasa.model import DEPTHS
from abhyasa.tutor import Tutor, Session, TUTOR_SP, MAX_TRIES
from abhyasa.llm import ask_json, rishi_ask
from abhyasa import demo as _demo   # registers the demo hook of the tutor

## Use

Leela shows ipywidgets, so the card is a widget in the output of the cell.
Leela keeps cell metadata that it does not know. Thus the cells from `write_exercise` keep their `metadata.abhyasa`.

```python
from abhyasa.nb import learn, exercise
learn('reciprocal-rank-fusion')      # a question card
ex = exercise('reciprocal-rank-fusion')
def rrf(lists, k=60): ...            # your code, in the next cell
ex.check(rrf)                        # runs the tests, records the attempt
```

## The tutor

`tutor` makes the tutor of the notebook one time. A call with a store or a model makes a new one.

In [ ]:
#| export
_st = AttrDict(tutor=None)

def tutor(learning=None, ask=None) -> Tutor:
    "The tutor of the notebook, made one time: the default vault and `rishi_ask`, unless you give others."
    if _st.tutor is None or learning is not None or ask is not None:
        _st.tutor = Tutor(learning or Learning(os.getenv('ABHYASA_VAULT') or os.getenv('VISHALAKSHI_VAULT')), ask or rishi_ask())
    return _st.tutor

def _esc(s): return html.escape(s or '').replace('\n', '<br>')

## The question card

`Card` has a box for the answer, a box for the think-aloud, a confidence slider and four buttons.

In [ ]:
#| export
class Card:
    "A question card widget over a `Session`: answer, think aloud, confidence, hint, and next."
    def __init__(self, sess:Session):
        import ipywidgets as w
        self.s, self.w = sess, w
        self.q = w.HTMLMath(); self.fb = w.HTMLMath()
        self.ans = w.Textarea(placeholder='Your answer', layout=w.Layout(width='100%', height='70px'))
        self.think = w.Textarea(placeholder='Think aloud: how you got there, what you are unsure of', layout=w.Layout(width='100%', height='90px'))
        self.conf = w.IntSlider(50, 0, 100, description='Confidence')
        b = lambda t, f, st='': (x := w.Button(description=t, button_style=st), x.on_click(lambda _: self._do(f)), x)[-1]
        self.buttons = w.HBox([b('Send', self._send, 'primary'), b('Hint', self._hint), b("I don't know", self._idk), b('Next', self._next)])
        self.demo = w.Output()
        self.box = w.VBox([self.q, self.ans, self.think, self.conf, self.buttons, self.fb, self.demo])
        self._next()

    def _do(self, f):
        try: f()
        except Exception as e: self.fb.value = f'<b>Error</b>: {_esc(str(e))}'

    def _show(self, q):
        if q is None: self.q.value, self.fb.value = '<b>Nothing to learn or review right now.</b>', ''; return
        opts = ''.join(f'<li>{i}. {_esc(o)}</li>' for i, o in enumerate(q.options)) if q.get('options') else ''
        self.q.value = (f"<small>{_esc(q.name)} · {DEPTHS[q.depth]} · {_esc(q.why)}</small><p style='font-size:1.2em'>{_esc(q.question)}</p>"
                        + (f'<ul style="list-style:none">{opts}</ul><small>Answer with the number.</small>' if opts else ''))
        self.ans.value = self.think.value = self.fb.value = ''

    def _send(self):
        r = self.s.answer(self.ans.value, self.think.value, self.conf.value)
        self.fb.value = fb_html(r); self._demo(r)
    def _demo(self, r):
        from IPython.display import display
        self.demo.clear_output()
        if (v := demo_view(r, self.s.tutor.learning)):
            with self.demo: display(v)
    def _hint(self): self.fb.value = f'<i>Hint:</i> {_esc(self.s.hint())}'
    def _idk(self): r = self.s.answer(dont_know=True); self.fb.value = fb_html(r); self._demo(r)
    def _next(self): self._show(self.s.next())
    def _ipython_display_(self):
        from IPython.display import display
        display(self.box)

`fb_html` shows a result in the card. `learn` starts a session in a card.

In [ ]:
#| export
def fb_html(r) -> str:
    "A result as HTML."
    if not r.done: return f"<b>Not yet.</b> {_esc(r.feedback)}<br><small>{r.tries_left} more tries.</small>"
    parts = [f"<b>{r.outcome}</b>", _esc(r.feedback), _esc(r.lesson), f"<small>Floor now {DEPTHS[r.floor]}. Press Next.</small>"]
    return '<br><br>'.join(p for p in parts if p)

def learn(focus:str=None, subject:str=None, limit:int=12) -> Card:
    "A question card for a session, on one concept when you name it."
    return Card(Session(tutor(), focus=focus, subject=subject, limit=limit))

After a miss on a concept that has a demo, the card shows the demo under the feedback. `demo_view` gives the demo as a notebook output. It uses `show` from rupaka, and leela draws the output in a sandboxed frame.

In [ ]:
#| export
def demo_view(r, learning):
    "The demo of result `r` as a notebook output, or None."
    from rupaka.render import show
    return show(d.spec) if r.get('demo') and (d := learning.demo(r.demo)) else None

A result with a demo gives an output with the rupaka mime and a full HTML page. A result without one gives nothing.

In [ ]:
from abhyasa.store import temp_learning
from rupaka.spec import gallery
from rupaka.render import MIME
dl = temp_learning(); dl.put('Vowel sandhi', 'Vowels join by rule.', 'sanskrit')
dl.put_demo(gallery()['vowel-sandhi'], 'vowel-sandhi')
mb = demo_view(AttrDict(demo='vowel-sandhi'), dl)._repr_mimebundle_()
assert MIME in mb and '<script' in mb['text/html']
test_eq(demo_view(AttrDict(correct=True), dl), None)

## Code exercises

The model writes a task (a small function to make), a stub and `assert` tests. The tests run on your function in the kernel.
The exercise shows a failing test, but not the fix. A pass at depth 4 (apply) counts as a pass in the learner model.

`CODE_SP` and `code_prompt` ask the model for one exercise.

In [ ]:
#| export
CODE_SP = TUTOR_SP + """
You write one small Python exercise that makes the learner apply a concept by implementing one function.
- The function is 5-25 lines for a correct solution, standard library and numpy only.
- Tests are 3-6 plain `assert` lines that call the function by name; include one edge case.
- Never put the solution in the stub."""

def code_prompt(c):
    return (f"Concept: {c['name']} (subject: {c.get('subject') or 'general'})\nStatement: {c.get('statement', '')}\n"
            f"Misconceptions seen: {'; '.join(c.get('misconceptions') or []) or 'none'}\n\n"
            'Return JSON: {"task": str, "fname": str, "stub": str, "tests": [str], "solution": str}')

The model writes the tests, so the notebook does not trust them. `safe_test` accepts only one `assert`.
The assert can use only the function of the learner, literals, comparisons, arithmetic and a short list of safe builtins.
It cannot use an import, a name or an attribute with a leading underscore, `open`, `exec` or `eval`.
`validate_exercise` checks the shape of the reply of the model. A test runs with only the safe builtins.
When a test runs on the main thread (as in a kernel), it also has a time limit. Thus an endless loop fails the test.

In [ ]:
#| export
import ast, builtins, signal, threading
from contextlib import contextmanager
SAFE_NAMES = ('abs', 'all', 'any', 'bool', 'dict', 'enumerate', 'float', 'int', 'isinstance', 'len', 'list', 'max', 'min',
              'range', 'reversed', 'round', 'set', 'sorted', 'str', 'sum', 'tuple', 'zip', 'True', 'False', 'None')
SAFE_BUILTINS = {n: getattr(builtins, n) for n in SAFE_NAMES if hasattr(builtins, n)}
SAFE_NODES = (ast.Module, ast.Assert, ast.Expr, ast.Compare, ast.Call, ast.Name, ast.Load, ast.Constant, ast.List, ast.Tuple, ast.Dict,
              ast.Set, ast.BinOp, ast.UnaryOp, ast.BoolOp, ast.Subscript, ast.Slice, ast.keyword, ast.Attribute, ast.IfExp,
              ast.operator, ast.unaryop, ast.boolop, ast.cmpop, ast.ListComp, ast.GeneratorExp, ast.comprehension, ast.Store)
TEST_SECONDS = 2.

def safe_test(t:str, fname:str) -> bool:
    "Whether `t` is one plain `assert` that uses only `fname`, literals and the safe builtins."
    try: tree = ast.parse(t)
    except SyntaxError: return False
    if len(tree.body) != 1 or not isinstance(tree.body[0], ast.Assert): return False
    bound = {n.id for n in ast.walk(tree) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store)}   # names of a comprehension
    for n in ast.walk(tree):
        if not isinstance(n, SAFE_NODES): return False
        if isinstance(n, ast.Name) and n.id not in (fname, *SAFE_NAMES, *bound): return False
        if isinstance(n, ast.Attribute) and n.attr.startswith('_'): return False
        if isinstance(n, ast.Constant) and isinstance(n.value, str) and len(n.value) > 2000: return False
    return True

def validate_exercise(r) -> AttrDict:
    "The model's exercise, checked: task, fname, stub and 1-12 safe tests. Raise `ValueError` with the reason."
    if not isinstance(r, dict): raise ValueError('the reply is not a JSON object')
    task, fname, stub, tests = r.get('task'), r.get('fname'), r.get('stub'), r.get('tests')
    if not (isinstance(task, str) and task.strip()): raise ValueError('`task` must be a text')
    if not (isinstance(fname, str) and fname.isidentifier() and not fname.startswith('_')): raise ValueError('`fname` must be a plain function name')
    if not isinstance(stub, str): raise ValueError('`stub` must be a text')
    if not (isinstance(tests, list) and 1 <= len(tests) <= 12 and all(isinstance(t, str) for t in tests)): raise ValueError('`tests` must be a list of 1 to 12 texts')
    if (bad := [t for t in tests if not safe_test(t, fname)]): raise ValueError(f'a test is not one plain assert on `{fname}`: {bad[0][:120]}')
    return AttrDict(task=task, fname=fname, stub=stub, tests=tests, solution=r.get('solution', '') if isinstance(r.get('solution'), str) else '')

@contextmanager
def time_limit(secs:float=TEST_SECONDS):
    "Raise `TimeoutError` after `secs`, on the main thread; elsewhere no limit is possible, so none is set."
    if threading.current_thread() is not threading.main_thread(): yield; return
    def boom(*_): raise TimeoutError(f'the test ran longer than {secs:g} s')
    old = signal.signal(signal.SIGALRM, boom); signal.setitimer(signal.ITIMER_REAL, secs)
    try: yield
    finally: signal.setitimer(signal.ITIMER_REAL, 0); signal.signal(signal.SIGALRM, old)

`Exercise` keeps one exercise open on the tutor. `check` grades it with the same tries and lessons as a question.

In [ ]:
#| export
class Exercise:
    "One code exercise on a concept. `check(fn)` runs its tests on your function."
    def __init__(self, tut:Tutor, slug:str):
        c = tut.learning.concept(slug)
        if c is None: raise KeyError(f'no concept {slug!r}')
        p = code_prompt(c)
        try: r = validate_exercise(ask_json(tut.ask, p, CODE_SP))
        except ValueError as e:   # one more try, with the reason
            r = validate_exercise(ask_json(tut.ask, f'{p}\n\nYour last reply was refused: {e}. Follow the rules exactly.', CODE_SP))
        self.t, self.c = tut, c
        self.q = AttrDict(id=uuid.uuid4().hex[:10], concept=slug, name=c.name, statement=c.statement, depth=4, kind='code',
                          question=r.task, answer=r.solution, rubric=r.tests, hints=[], tries=[], session='', why='code exercise')
        self.fname, self.stub, self.tests = r.fname, r.stub, r.tests
        tut.open[self.q.id] = self.q

    def _ipython_display_(self):
        from IPython.display import display, Markdown
        display(Markdown(f"**{self.c.name}** · code exercise\n\n{self.q.question}\n\n```python\n{self.stub}\n```\n\n"
                         f"Write `{self.fname}` in a cell, then run `ex.check({self.fname})`."))

    def run(self, fn):
        "Run the tests on `fn`, and return the failures as `(test, error)`."
        fails = []
        for t in self.tests:
            try:
                with time_limit(): exec(t, {'__builtins__': SAFE_BUILTINS, self.fname: fn})
            except Exception as e: fails.append((t, f'{type(e).__name__}: {e}'.strip()))
        return fails

    def check(self, fn, confidence:int=70):
        "Run the tests. A fail shows the first failing test. After the last try, the tutor teaches."
        if self.q.id not in self.t.open: return 'This exercise is closed. Make a new one with `exercise(...)`.'
        fails = self.run(fn)
        g = AttrDict(correct=not fails, grade=10 * (len(self.tests) - len(fails)) // len(self.tests),
                     first_error=f'This test fails: `{fails[0][0]}` ({fails[0][1]})' if fails else '', misconception='', feedback='')
        if fails and len(self.q.tries) + 1 < MAX_TRIES:
            self.q.tries.append(dict(answer=f'{len(fails)} failing', first_error=g.first_error, confidence=confidence))
            return f"{len(self.tests) - len(fails)} of {len(self.tests)} tests pass. {g.first_error}"
        if fails: self.q.tries.append(dict(answer=f'{len(fails)} failing', first_error=g.first_error, confidence=confidence))
        r = self.t._close(self.q, not fails, self.q.tries[0]['confidence'] if fails else confidence, g, teach=bool(fails))
        return fb_text(r)

    def hint(self) -> str: return self.t.hint(self.q.id)

`fb_text` shows a result as text. `exercise` makes an exercise on the tutor of the notebook.

In [ ]:
#| export
def fb_text(r): return '\n\n'.join(o for o in (r.outcome, r.feedback, r.lesson, f'Floor now {DEPTHS[r.floor]}.') if o)

def exercise(slug:str) -> Exercise:
    "A code exercise on concept `slug`."
    return Exercise(tutor(), slug)

## Exercise notebooks

`write_exercise` writes the exercise into tagged cells of a new notebook file, for you to open later.
It never writes to a notebook that is already there.

In [ ]:
#| export
def write_exercise(ex:Exercise, path) -> Path:
    "A new notebook at `path` with the task, a stub cell and the check. Do not write over a file."
    p = Path(path).expanduser()
    if p.exists(): raise FileExistsError(p)
    tag = dict(abhyasa=dict(concept=ex.q.concept, qid=ex.q.id))
    cells = [mk_cell(f"# {ex.c.name}\n\n{ex.q.question}", 'markdown', metadata=tag),
             mk_cell(ex.stub, metadata=tag),
             mk_cell('\n'.join(ex.tests), metadata=dict(abhyasa=dict(concept=ex.q.concept, qid=ex.q.id, tests=True)))]
    write_nb(new_nb(cells), p)
    return p

## Tests

The first tests check the safety rules. Then a fake model writes a full exercise.

`safe_test` refuses an import, a name with two leading underscores, a call to anything but the function and the safe builtins, and code that is not one assert.

In [ ]:
assert safe_test("assert rrf([['a'], ['b']], k=0) == ['a', 'b']", 'rrf')
assert safe_test("assert all(x in 'ab' for x in rrf([['a', 'b']]))", 'rrf')
for bad in ("import os", "assert __import__('os')", "assert rrf.__globals__", "assert open('/etc/passwd')", "assert exec('1')",
            "rrf([])", "while True: pass", "assert rrf(1); import os", "assert (lambda: 1)()"):
    assert not safe_test(bad, 'rrf'), bad

`validate_exercise` refuses an empty or a text `tests`, a missing name, and an unsafe test, with a reason.

In [ ]:
good = dict(task='t', fname='f', stub='def f(): ...', tests=['assert f() == 1'])
test_eq(validate_exercise(good).fname, 'f')
for bad, why in ((dict(good, tests=[]), 'list of 1 to 12'), (dict(good, tests='assert f()'), 'list of 1 to 12'),
                 ({k: v for k, v in good.items() if k != 'fname'}, 'fname'), (dict(good, tests=['import os']), 'not one plain assert'), ([], 'JSON object')):
    test_fail(lambda: validate_exercise(bad), contains=why)

A test that runs too long fails after the time limit, and the kernel continues. Off the main thread, a limit is not possible, so the code sets none.

In [ ]:
import threading, time
if threading.current_thread() is threading.main_thread():   # a kernel runs cells here, so the limit works
    t0 = time.time()
    try:
        with time_limit(.2):
            for _ in range(10**10): pass   # long, but never endless, so no test runner can hang on it
        raise AssertionError('no timeout')
    except TimeoutError: assert time.time() - t0 < 2
else:   # nbdev-test runs cells on another thread: no signal is possible there, so the limit steps aside
    with time_limit(.2): pass

A fake model writes an exercise with two tests. The tutor of the notebook uses a temporary shelf.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
lr = temp_learning()
lr.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank).', 'ml')
fake = Fake({'Return JSON: {"task"': dict(task='Write rrf(lists, k) that returns items by fused score.', fname='rrf',
             stub='def rrf(lists, k=60):\n    ...', tests=["assert rrf([['a','b'],['b','a']])[0] in 'ab'", "assert rrf([['a'],['b']], k=0) == ['a','b']"],
             solution='...'), 'Teach the smallest step': 'Sum 1/(k+r). Check: what if k is large?'},
            dflt=dict(question='Why ranks?', answer='Comparable.', rubric=['comparable']))
t = tutor(lr, fake)
ex = exercise('reciprocal-rank-fusion')

A wrong function fails both tests, and the exercise stays open.

In [ ]:
def rrf(lists, k=60): return []
assert ex.check(rrf).startswith('0 of 2 tests pass')

A correct function passes after the failed try, so the outcome is `fragile` and the concept is `shaky` at depth 4.

In [ ]:
def rrf(lists, k=60):
    s = {}
    for l in lists:
        for r, x in enumerate(l): s[x] = s.get(x, 0) + 1 / (k + r + 1)
    return sorted(s, key=lambda x: (-s[x], x))
out = ex.check(rrf)
assert out.startswith('fragile'), out     # passed after a failed try
test_eq(lr.concept('reciprocal-rank-fusion').shaky, 4)

`write_exercise` tags the cells and does not write over a file.

In [ ]:
import tempfile
from fastcore.nbio import read_nb
ex2 = exercise('reciprocal-rank-fusion')
p = write_exercise(ex2, Path(tempfile.mkdtemp())/'rrf.ipynb')
nb = read_nb(p)
test_eq(nb.cells[1].metadata['abhyasa']['concept'], 'reciprocal-rank-fusion')
test_fail(lambda: write_exercise(ex2, p), exc=FileExistsError)

`learn` shows a card with the first question.

In [ ]:
c = learn('reciprocal-rank-fusion')
assert 'Why ranks?' in c.q.value

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()